# ML-Leaks Model Independent Membership Inference Attack

목표: `08_ML_Leaks/data` 폴더 구조를 그대로 사용하여 합성 이벤트 로그에 대한 **Model Independent MIA**를 실행한다.

기본 경로 구조:

```text
08_ML_Leaks/
├─ 08_02_Model_Independent/
│  └─ model_independent_mia.ipynb
└─ data/
   ├─ Real/
   │  ├─ mimicel_train.csv
   │  ├─ mimicel_val.csv
   │  └─ mimicel_test.csv
   ├─ Rule_based/
   ├─ ProcessGAN/
   └─ PALSYN/
```

이 노트북은 우선 `Rule_based` 합성 로그 하나를 대상으로 실행하도록 설정되어 있다.


## 구현 방식 요약

ML-Leaks의 model independent 설정은 공격자가 target model의 구조를 모른다고 가정한다. 따라서 shadow model을 target model과 같은 구조로 고정하지 않고, 서로 다른 알고리즘의 shadow classifier를 여러 개 학습한 뒤 이들의 posterior 출력으로 attack model을 학습한다.

여기서는 다음 방식으로 구현한다.

1. 합성 로그로 target utility model 학습
2. real validation set을 `shadow_in`, `shadow_out`으로 분할
3. RF, ExtraTrees, LogisticRegression, MLP를 shadow classifier로 학습
4. 각 shadow classifier의 posterior에서 top-k 확률, entropy, confidence gap 추출
5. `shadow_in = member`, `shadow_out = non-member`로 attack model 학습
6. target utility model에 `real_train`과 `real_test`를 넣어 posterior 추출
7. attack model이 `real_train`은 member, `real_test`는 non-member로 맞히는지 평가

출력 지표: `MIA AUC`, `Accuracy`, `Precision`, `Recall`


In [20]:
from pathlib import Path

# run_rule_based_all_seeds.py
# Model Independent ML-Leaks for Rule-based seed41~45

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

CASE_COL = "stay_id"
ACT_COL = "activity"
TIME_COL = "timestamps"

DATA_DIR = Path("../data")
REAL_DIR = DATA_DIR / "Real"
RULE_DIR = DATA_DIR / "Rule_based"
RESULT_DIR = Path("./results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "los_over_4h"
RANDOM_STATE = 42

RULE_BASED_FILES = [
    "rule_based_synthetic_data_seed41.csv",
    "rule_based_synthetic_data_seed42.csv",
    "rule_based_synthetic_data_seed43.csv",
    "rule_based_synthetic_data_seed44.csv",
    "rule_based_synthetic_data_seed45.csv",
]

def load_event_log(path):
    df = pd.read_csv(path)
    df = df.loc[:, ~df.columns.duplicated()].copy()
    if TIME_COL in df.columns:
        df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    return df

def build_case_features(df):
    g = df.groupby(CASE_COL)

    base = pd.DataFrame(index=g.size().index)
    base["trace_length"] = g.size()

    time_agg = g[TIME_COL].agg(["min", "max"])
    base["duration_minutes"] = (
        (time_agg["max"] - time_agg["min"]).dt.total_seconds() / 60
    ).fillna(0)

    act_count = pd.crosstab(df[CASE_COL], df[ACT_COL]).add_prefix("act_count__")

    act_ratio = act_count.div(act_count.sum(axis=1).replace(0, 1), axis=0)
    act_ratio.columns = [c.replace("act_count__", "act_ratio__") for c in act_ratio.columns]

    ordered = df.sort_values([CASE_COL, TIME_COL])
    first_last = ordered.groupby(CASE_COL)[ACT_COL].agg(["first", "last"])
    first_last.columns = ["first_activity", "last_activity"]

    out = (
        base.join(act_count)
        .join(act_ratio)
        .join(first_last)
        .reset_index()
    )
    return out

def make_label(df):
    return (df["duration_minutes"] >= 240).astype(int)

def align_columns(*dfs):
    cols = sorted(set().union(*[set(x.columns) for x in dfs]))
    return [x.reindex(columns=cols) for x in dfs]

def make_preprocessor(X):
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]

    return ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ])

def topk_features(proba):
    sorted_p = -np.sort(-proba, axis=1)
    if sorted_p.shape[1] < 3:
        sorted_p = np.hstack([sorted_p, np.zeros((len(sorted_p), 3-sorted_p.shape[1]))])
    top = sorted_p[:, :3]
    entropy = -(proba * np.log(proba + 1e-12)).sum(axis=1, keepdims=True)
    gap = (top[:, [0]] - top[:, [1]])
    return np.hstack([top, entropy, gap])

real_train = build_case_features(load_event_log(REAL_DIR / "mimicel_train.csv"))
real_val = build_case_features(load_event_log(REAL_DIR / "mimicel_val.csv"))
real_test = build_case_features(load_event_log(REAL_DIR / "mimicel_test.csv"))

results = []

for file_name in RULE_BASED_FILES:

    syn = build_case_features(load_event_log(RULE_DIR / file_name))

    y_syn = make_label(syn)
    y_val = make_label(real_val)

    X_syn = syn.drop(columns=[CASE_COL], errors="ignore")
    X_val = real_val.drop(columns=[CASE_COL], errors="ignore")
    X_train = real_train.drop(columns=[CASE_COL], errors="ignore")
    X_test = real_test.drop(columns=[CASE_COL], errors="ignore")

    X_syn, X_val, X_train, X_test = align_columns(
        X_syn, X_val, X_train, X_test
    )

    target = Pipeline([
        ("prep", make_preprocessor(X_syn)),
        ("clf", RandomForestClassifier(
            n_estimators=400,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
    target.fit(X_syn, y_syn)

    X_in, X_out, y_in, y_out = train_test_split(
        X_val, y_val,
        test_size=0.5,
        random_state=RANDOM_STATE,
        stratify=y_val
    )

    attack_X = []
    attack_y = []

    shadow_models = [
        RandomForestClassifier(n_estimators=300, random_state=42),
        ExtraTreesClassifier(n_estimators=300, random_state=42),
        LogisticRegression(max_iter=1000),
        MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=42)
    ]

    for shadow in shadow_models:

        pipe = Pipeline([
            ("prep", make_preprocessor(X_in)),
            ("clf", shadow)
        ])

        pipe.fit(X_in, y_in)

        p_in = pipe.predict_proba(X_in)
        p_out = pipe.predict_proba(X_out)

        attack_X.append(topk_features(p_in))
        attack_y.append(np.ones(len(p_in)))

        attack_X.append(topk_features(p_out))
        attack_y.append(np.zeros(len(p_out)))

    attack_X = np.vstack(attack_X)
    attack_y = np.concatenate(attack_y)

    attack_model = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )
    attack_model.fit(attack_X, attack_y)

    member = topk_features(target.predict_proba(X_train))
    nonmember = topk_features(target.predict_proba(X_test))

    X_eval = np.vstack([member, nonmember])
    y_eval = np.concatenate([
        np.ones(len(member)),
        np.zeros(len(nonmember))
    ])

    score = attack_model.predict_proba(X_eval)[:,1]
    pred = attack_model.predict(X_eval)

    results.append({
        "file": file_name,
        "mia_auc": roc_auc_score(y_eval, score),
        "mia_accuracy": accuracy_score(y_eval, pred),
        "mia_precision": precision_score(y_eval, pred, zero_division=0),
        "mia_recall": recall_score(y_eval, pred, zero_division=0),
    })

summary = pd.DataFrame(results)
summary.to_csv(
    RESULT_DIR / "model_independent_mia_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print(summary)
print(summary[["mia_auc"]].mean())


                                   file   mia_auc  mia_accuracy  \
0  rule_based_synthetic_data_seed41.csv  0.464748      0.436474   
1  rule_based_synthetic_data_seed42.csv  0.510481      0.456353   
2  rule_based_synthetic_data_seed43.csv  0.463030      0.449438   
3  rule_based_synthetic_data_seed44.csv  0.493731      0.455488   
4  rule_based_synthetic_data_seed45.csv  0.471361      0.443388   

   mia_precision  mia_recall  
0       0.754639    0.407119  
1       0.783613    0.414905  
2       0.750000    0.437152  
3       0.778468    0.418242  
4       0.744722    0.431591  
mia_auc    0.48067
dtype: float64
